# 第三章：PyTorch 的主要组成模块

这一章讲的是一个深度学习项目最常见的流水线：配置参数、读取数据、构建模型、初始化参数、定义损失函数、选择优化器、训练与评估、可视化。

这个 notebook 用一个很小的合成分类任务，把完整流程跑一遍。每个单元都可以独立理解，按顺序运行即可。

## 1. 导入库并固定随机种子

固定随机种子是为了让每次运行结果尽量一致，方便排查问题。

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
import matplotlib.pyplot as plt

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

# 这里用 CPU 演示，兼容性最好；真实训练时可以按需切换到 cuda/mps。
device = torch.device("cpu")
print("使用设备：", device)


## 2. 基本配置

实际项目里，常把超参数集中放在一个字典里，便于统一修改。

In [ ]:
config = {
    "num_samples": 256,
    "batch_size": 32,
    "learning_rate": 0.05,
    "epochs": 6,
    "num_classes": 2,
}

config


## 3. 构造一个小数据集

为了不用下载数据，我们合成 8x8 的“灰度图片”。规则很简单：上半部分平均值大于下半部分时标记为 1，否则标记为 0。

In [ ]:
num_samples = config["num_samples"]
images = torch.randn(num_samples, 1, 8, 8)
upper_mean = images[:, :, :4, :].mean(dim=(1, 2, 3))
lower_mean = images[:, :, 4:, :].mean(dim=(1, 2, 3))
labels = (upper_mean > lower_mean).long()

print("图片张量形状：", images.shape)
print("标签张量形状：", labels.shape)
print("前 10 个标签：", labels[:10].tolist())


## 4. Dataset 和 DataLoader

`Dataset` 管“有哪些样本”，`DataLoader` 管“每次取多少、是否打乱、如何组成 batch”。

In [ ]:
dataset = TensorDataset(images, labels)
train_dataset, val_dataset = random_split(
    dataset,
    [200, 56],
    generator=torch.Generator().manual_seed(seed),
)

train_loader = DataLoader(train_dataset, batch_size=config["batch_size"], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=config["batch_size"], shuffle=False)

batch_images, batch_labels = next(iter(train_loader))
print("一个 batch 的图片形状：", batch_images.shape)
print("一个 batch 的标签形状：", batch_labels.shape)


## 5. 可视化一个样本

在图像任务里，先看一眼数据很重要。不要盲目把数据丢进模型。

In [ ]:
plt.figure(figsize=(3, 3))
plt.imshow(images[0, 0], cmap="gray")
plt.title(f"label={labels[0].item()}")
plt.axis("off")
plt.show()


## 6. 构建模型

`nn.Module` 是 PyTorch 模型的基类。通常在 `__init__` 里定义层，在 `forward` 里定义数据怎么流动。

In [ ]:
class TinyClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1 * 8 * 8, 16),
            nn.ReLU(),
            nn.Linear(16, config["num_classes"]),
        )

    def forward(self, x):
        return self.net(x)

model = TinyClassifier().to(device)
print(model)


## 7. 参数初始化

初始化会影响模型训练起点。这里对全连接层使用 Xavier 初始化，bias 置零。

In [ ]:
def init_weights(module):
    if isinstance(module, nn.Linear):
        nn.init.xavier_uniform_(module.weight)
        nn.init.zeros_(module.bias)

model.apply(init_weights)

first_linear = model.net[1]
print("第一层权重均值：", first_linear.weight.mean().item())
print("第一层权重标准差：", first_linear.weight.std().item())


## 8. 损失函数和优化器

分类任务常用 `CrossEntropyLoss`。优化器负责根据梯度更新参数。

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=config["learning_rate"])

logits = model(batch_images.to(device))
loss = criterion(logits, batch_labels.to(device))
print("模型输出形状：", logits.shape)
print("一个 batch 的初始 loss：", loss.item())


## 9. 训练与评估函数

训练模式会计算梯度并更新参数；评估模式只前向推理，不更新参数。

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
    return total_loss / len(loader.dataset)


def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    correct = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            total_loss += loss.item() * x.size(0)
            pred = logits.argmax(dim=1)
            correct += (pred == y).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)


## 10. 开始训练

观察 loss 是否下降、accuracy 是否上升，是训练是否正常的第一层信号。

In [ ]:
history = {"train_loss": [], "val_loss": [], "val_acc": []}

for epoch in range(1, config["epochs"] + 1):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc = evaluate(model, val_loader, criterion)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    print(f"epoch {epoch}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}, val_acc={val_acc:.2%}")


## 11. 可视化训练曲线

曲线比单个数字更容易看出趋势。

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(history["train_loss"], label="train_loss")
plt.plot(history["val_loss"], label="val_loss")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.title("训练/验证损失曲线")
plt.show()


## 12. 优化器状态

优化器除了保存学习率，还会保存动量等内部状态。SGD 没有动量时状态较少，Adam 的状态会更多。

In [ ]:
print("优化器参数组：")
for group in optimizer.param_groups:
    print({"lr": group["lr"], "params_count": len(group["params"])})

print("优化器 state 条目数：", len(optimizer.state))
